In [ ]:
import pygame
import numpy as np
import sys
image = pygame.image.load("map.jpg")
fog_of_war = pygame.Surface(image.get_size(), pygame.SRCALPHA)
fog_of_war.fill((0, 0, 0, 255))
screen = pygame.display.set_mode(image.get_size())
screen.blit(image, (0, 0))
screen.blit(fog_of_war, (0, 0))
pygame.display.flip()
while True:
    mouse_pos = pygame.mouse.get_pos()
    if pygame.mouse.get_pressed()[0]:
        fog_of_war.set_alpha(0)
        pygame.draw.circle(fog_of_war, (0, 0, 0, 0), mouse_pos, 50)
        fog_of_war.set_alpha(255)
    screen.blit(image, (0, 0))
    screen.blit(fog_of_war, (0, 0))
    pygame.display.flip()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

In [ ]:
import pygame
import numpy as np
import sys


def main():
    # Load the image:
    image = pygame.image.load("map.jpg")

    # Create a fog of war surface:
    fog_of_war = pygame.Surface(image.get_size(), pygame.SRCALPHA)
    fog_of_war.fill((0, 0, 0, 255))

    # Create two screens:
    screen = pygame.display.set_mode(image.get_size())
    screen_gm = pygame.display.set_mode(image.get_size())

    # Display the image and the fog of war surface on both screens:
    screen.blit(image, (0, 0))
    screen.blit(fog_of_war, (0, 0))
    pygame.display.flip()

    screen_gm.blit(image, (0, 0))
    screen_gm.blit(fog_of_war, (0, 0))
    pygame.display.flip()

    # Create a main loop:
    while True:

        # Get the mouse position:
        mouse_pos = pygame.mouse.get_pos()

        # Erase the fog of war around the mouse cursor on click on the player screen:
        if pygame.mouse.get_pressed()[0]:
            fog_of_war.set_alpha(0)
            pygame.draw.circle(fog_of_war, (0, 0, 0, 0), mouse_pos, 50)
            fog_of_war.set_alpha(255)

        # Display the image and the fog of war surface on both screens:
        screen.blit(image, (0, 0))
        screen.blit(fog_of_war, (0, 0))
        pygame.display.flip()

        screen_gm.blit(image, (0, 0))
        screen_gm.blit(fog_of_war, (0, 0))
        pygame.display.flip()

        # Handle events:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()


if __name__ == "__main__":
    main()

In [ ]:
import cv2
import numpy as np


def load_image(image_path):
    image = cv2.imread(image_path)
    return image


def create_fog_of_war(image_size):
    fog_of_war = cv2.rectangle(np.zeros(image_size, dtype=np.uint8), (0, 0), (image_size[1], image_size[0]), (255, 255, 255), -1)
    # fog_of_war = cv2.zeros(image_size, dtype=cv2.uint8)
    return fog_of_war


def draw_fog_of_war(image, fog_of_war):
    cv2.rectangle(image, (0, 0), image.shape[:2], (0, 0, 0), -1)
    cv2.addWeighted(image, 0.5, fog_of_war, 0.5, 0, image)


def erase_fog_of_war(edit_window_image, mouse_position):
    cv2.circle(edit_window_image, mouse_position, 10, (255, 255, 255), -1)


def main():
    # Load the image and create two copies of it.
    image = load_image('map.jpg')
    display_window_image = image.copy()
    edit_window_image = image.copy()

    # Create a black image the same size as the original image.
    fog_of_war = create_fog_of_war(image.shape)

    # Draw a black rectangle over the entire display window image.
    draw_fog_of_war(display_window_image, fog_of_war)

    # Draw the edit window image over the display window image, but with a lower opacity.
    cv2.addWeighted(display_window_image, 0.5, edit_window_image, 0.5, 0, display_window_image)

    # Start a loop that listens for mouse events.
    while True:

        # Get the mouse position.
        mouse_position = cv2.getWindowImageRect('Display Window')[2:4]

        # When a mouse click event is detected, erase the fog of war at that point.
        if cv2.getWindowProperty('Display Window', cv2.WND_PROP_AUTOSIZE) == cv2.WINDOW_AUTOSIZE_ON:
            erase_fog_of_war(edit_window_image, mouse_position)

        # Update the display window image to match the edit window image.
        display_window_image[:] = edit_window_image[:]

        # Display the display window image.
        cv2.imshow('Display Window', display_window_image)

        # If the user presses the 'q' key, break out of the loop.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()


if __name__ == '__main__':
    main()

In [ ]:
import cv2
import numpy as np
import keyboard
from tkinter import filedialog
import tkinter as tk

# Initialize flags and variables
drawing = False
opacity = 0.8


def update_fog_of_war(event, x, y, flags, param):
    global drawing

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        cv2.circle(fog_of_war, (x, y), 30, (1, 1, 1), -1)
        cv2.circle(edit_fog_of_war, (x, y), 30, (1, 1, 1), -1)
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            cv2.circle(fog_of_war, (x, y), 30, (1, 1, 1), -1)
            cv2.circle(edit_fog_of_war, (x, y), 30, (1, 1, 1), -1)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False


# def ResizeWithAspectRatio(image, width=None, height=None, inter=cv2.INTER_AREA):
#     dim = None
#     (h, w) = image.shape[:2]

#     if width is None and height is None:
#         return image
#     if width is None:
#         r = height / float(h)
#         dim = (int(w * r), height)
#     else:
#         r = width / float(w)
#         dim = (width, int(h * r))

#     return cv2.resize(image, dim, interpolation=inter)


def display(image, windows_name):
    wdn_size = cv2.getWindowImageRect(windows_name)[2:4]
    ratio = min(wdn_size[0] / image.shape[1], wdn_size[1] / image.shape[0])
    # print(windows_name, image.shape, wdn_size, ratio)
    # cv2.resize(image, (int(image.shape[1] * ratio), int(image.shape[0] * ratio)), interpolation=cv2.INTER_AREA)
    # cv2.resize(image, wdn_size, interpolation=cv2.INTER_AREA)
    cv2.imshow(windows_name, image)


# Load the image (map) and create fog of war
map_image = cv2.imread("map.jpg")
image_size = map_image.shape
fog_of_war = np.zeros(image_size, dtype=np.uint8)
edit_fog_of_war = np.full(image_size, 5, dtype=np.uint8)


# Create windows for display and editing
cv2.namedWindow("Fog of War", cv2.WINDOW_NORMAL)
cv2.namedWindow("Edit Fog of War", cv2.WINDOW_NORMAL)

# Set mouse callback for editing
cv2.setMouseCallback("Edit Fog of War", update_fog_of_war)

while True:
    # Combine the fog of war with the map image
    # result_edit = cv2.addWeighted(map_image, 1-opacity, fog_of_war, opacity, 0)
    result_edit = cv2.divide(map_image, edit_fog_of_war)
    # cv2.imshow("Edit Fog of War", result_edit)
    display(result_edit, "Edit Fog of War")

    result = cv2.multiply(map_image, fog_of_war)
    # cv2.imshow("Fog of War", result)
    display(result, "Fog of War")

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q") or key == 27:
        break

    # Handle 'Ctrl+L' to load a new image
    if keyboard.is_pressed('ctrl+l'):
        root = tk.Tk()
        root.withdraw()
        file_path = filedialog.askopenfilename(title="Select a new map image")
        if file_path:
            map_image = cv2.imread(file_path)
            image_size = map_image.shape
            fog_of_war = np.zeros(image_size, dtype=np.uint8)
            edit_fog_of_war = np.full(image_size, 5, dtype=np.uint8)

cv2.destroyAllWindows()